# Genetic, gene-set, and functional-correlation analysis

This notebook builds the correlation-analysis figures from the trait-level resources and the scDRS+ score folders.

Main outputs are generated for each dataset in `DATASET_CONFIGS`:

- `tms_facs` → `august_all/scdrs+_results_gram3`
- `ts_facs` → `ct_validations/ts_facs/scdrs+_results`
- `tms_droplet` → `ct_validations/tms_droplet/scdrs+_results`

For each dataset, the notebook writes CSV correlation matrices, CSV correlation-statistics tables, bootstrap summaries, the functional-vs-genetic split heatmap, and a five-panel scatter plot comparing functional correlation against genetic, absolute genetic, gene-level, pathway, and marginal correlations. Matching TSV files are retained for backward compatibility when `WRITE_LEGACY_TSV = True`.

Run all cells to regenerate the figures, CSV files, optional TSV files, and the final output manifest.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple
import itertools
import re
import warnings

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import seaborn as sns

try:
    from gprofiler import GProfiler
except ImportError:
    GProfiler = None

sns.set_context("notebook")


## Configuration


In [2]:
# === scDRS-FM reproduction: portable path anchor (injected, P5) ===
import os as _os
from pathlib import Path as _Path
def _find_repo_root():
    # 1) explicit override wins
    env = _os.environ.get('SCDRSFM_BASE')
    if env:
        return _Path(env)
    # 2) search upward from CWD for the reproduction repo root
    #    (a directory containing both 'scDRS-FM-main' and 'scripts')
    here = _Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / 'scDRS-FM-main').is_dir() and (d / 'scripts').is_dir():
            return d
    # 3) last resort: current working directory
    return here
BASE = _find_repo_root()
DATA = BASE / 'data'
RESULTS = BASE / 'results'
MAGMA_REF = BASE / 'magma_ref'
assert BASE.exists(), f'Repro root not found (set SCDRSFM_BASE to the repo root): {BASE}'
GENET_COR_CSV = DATA / 'genet_cor.csv'


In [3]:
# ---------------------------------------------------------------------
# Input paths
# ---------------------------------------------------------------------
GS_SPLIT_DIR = DATA / "gene_sets" / "gs_split"
LDSC_COR_DIR = DATA / "ldsc_genet_cor"  # unused in this reproduction (genetic corr read from genet_cor.csv); kept for API parity
MAGMA_GENE_ZSTAT_FILE = MAGMA_REF / "MAGMA_v108_GENE_10_ZSTAT_for_scDRS.txt"

# Outputs are grouped by dataset. CSV is the primary tabular format.
OUTPUT_ROOT = Path("correlation_analysis_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
GLOBAL_MATRIX_DIR = OUTPUT_ROOT / "global_matrices"
GLOBAL_MATRIX_DIR.mkdir(parents=True, exist_ok=True)

# Keep matching TSV files so existing downstream workflows continue to work.
WRITE_LEGACY_TSV = True


@dataclass(frozen=True)
class DatasetConfig:
    name: str
    score_dir: Path


DATASET_CONFIGS = [
    DatasetConfig("tms_facs", RESULTS / "real" / "tms_facs"),
    DatasetConfig("ts_facs", RESULTS / "real" / "ts_facs"),
    DatasetConfig("tms_droplet", RESULTS / "real" / "tms_droplet"),
]

# Trait files to ignore globally.
TRAITS_TO_DROP = {
    "PASS_Parkinsons23andMe_Corces2020",
}

# Pathway enrichment settings.
PATHWAY_SOURCES = ["GO:BP"]
MIN_PATHWAY_GENES = 5
PATHWAY_JACCARD_PRUNE_THRESHOLD = 0.30
PATHWAY_ENRICHMENT_CACHE = OUTPUT_ROOT / "gprofiler_pathway_enrichment.pkl"

# Bootstrap settings.
N_BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SAMPLE_FRAC = 0.80
RANDOM_SEED = 0

# Correlation/significance settings.
FUNCTIONAL_MC_N_CTRL = 1000
FUNCTIONAL_STAR_ALPHA = 0.05
GENETIC_STAR_ALPHA = 0.05

# Plotting.
FIG_DPI = 300


In [4]:
# Curated trait set used in the split functional/genetic heatmap.
# First 4 are brain/neuro traits, next 4 immune traits, remainder other traits.
SUBSET_TRAITS = [
    "PASS_BIP_Mullins2021",
    "PASS_Schizophrenia_Pardinas2018",
    "PASS_MDD_Howard2019",
    "UKB_460K.body_BMIz",
    "PASS_Rheumatoid_Arthritis",
    "PASS_IBD_deLange2017",
    "PASS_Multiple_sclerosis",
    "PASS_Lupus",
    "UKB_460K.blood_RBC_DISTRIB_WIDTH",
    "UKB_460K.biochemistry_Glucose",
    "PASS_Type_2_Diabetes",
    "PASS_AtrialFibrillation_Nielsen2018",
    "UKB_460K.bp_SYSTOLICadjMEDz",
    "UKB_460K.biochemistry_Cholesterol",
    "UKB_460K.biochemistry_LDLdirect",
    "UKB_460K.biochemistry_TotalProtein",
]

SUBSET_GROUPS = [
    ("Brain", "red", 4),
    ("Immune", "blue", 4),
    ("Other", "green", len(SUBSET_TRAITS) - 8),
]

TRAIT_NAME_DICT = {
    "PASS_BIP_Mullins2021": "Bipolar Disorder (BIP)",
    "PASS_Schizophrenia_Pardinas2018": "Schizophrenia (SCZ)",
    "PASS_MDD_Howard2019": "Major Depressive Disorder (MDD)",
    "UKB_460K.mental_NEUROTICISM": "Neuroticism",
    "PASS_Intelligence_SavageJansen2018": "Intelligence (IQ)",
    "PASS_Insomnia_Jansen2019": "Insomnia",
    "UKB_460K.body_BMIz": "Body Mass Index (BMI)",
    "PASS_Type_1_Diabetes": "Type 1 Diabetes (T1D)",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid Arthritis (RA)",
    "UKB_460K.disease_ASTHMA_DIAGNOSED": "Asthma",
    "PASS_IBD_deLange2017": "Inflammatory Bowel Disease (IBD)",
    "PASS_Multiple_sclerosis": "Multiple Sclerosis (MS)",
    "PASS_Lupus": "Lupus",
    "UKB_460K.blood_RBC_DISTRIB_WIDTH": "Red Blood Cell Distribution Width (RDW)",
    "UKB_460K.biochemistry_Glucose": "Glucose",
    "PASS_Type_2_Diabetes": "Type 2 Diabetes (T2D)",
    "PASS_AtrialFibrillation_Nielsen2018": "Atrial Fibrillation (AF)",
    "UKB_460K.bp_SYSTOLICadjMEDz": "Systolic Blood Pressure (SBP)",
    "UKB_460K.biochemistry_Cholesterol": "Total Cholesterol (TC)",
    "UKB_460K.biochemistry_LDLdirect": "Low-Density Lipoprotein Cholesterol (LDL)",
    "UKB_460K.biochemistry_TotalProtein": "Total Protein (TP)",
}

COMPARATOR_LABELS = {
    "genetic": "Genetic",
    "abs_genetic": "|Genetic|",
    "gene": "Gene-level",
    "pathway": "Pathway",
    "marginal": "Marginal",
}


## Utility functions


In [5]:
def list_trait_files(gs_split_dir: Path, traits_to_drop: Iterable[str] = ()) -> List[str]:
    """Return trait IDs from the scDRS gene-set split directory."""
    traits_to_drop = set(traits_to_drop)
    traits = [p.name for p in gs_split_dir.iterdir() if p.is_file() and not p.name.startswith(".")]
    traits = [t for t in traits if t not in traits_to_drop]
    return sorted(traits)


def ensure_output_dir(path: Path) -> Path:
    """Create an output directory and return it."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def export_dataframe_csv(
    df: pd.DataFrame,
    csv_path: Path,
    *,
    index: bool = False,
    index_label: Optional[str] = None,
    write_legacy_tsv: bool = WRITE_LEGACY_TSV,
) -> Dict[str, Path]:
    """Write a DataFrame to CSV and optionally to a matching legacy TSV file."""
    csv_path = Path(csv_path)
    if csv_path.suffix.lower() != ".csv":
        raise ValueError(f"CSV output path must end in .csv: {csv_path}")

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=index, index_label=index_label)
    written = {"csv": csv_path}

    if write_legacy_tsv:
        tsv_path = csv_path.with_suffix(".tsv")
        df.to_csv(tsv_path, sep="\t", index=index, index_label=index_label)
        written["tsv"] = tsv_path

    return written


def export_matrix_csv(
    matrix: pd.DataFrame,
    csv_path: Path,
    *,
    index_label: str = "trait",
    write_legacy_tsv: bool = WRITE_LEGACY_TSV,
) -> Dict[str, Path]:
    """Write a labeled matrix in wide CSV form, with traits preserved in the first column."""
    matrix_to_write = matrix.copy()
    matrix_to_write.index.name = index_label
    return export_dataframe_csv(
        matrix_to_write,
        csv_path,
        index=True,
        index_label=index_label,
        write_legacy_tsv=write_legacy_tsv,
    )


def clean_finite_pair(x: Sequence[float], y: Sequence[float]) -> Tuple[np.ndarray, np.ndarray]:
    """Return finite paired values from two arrays."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]


def pearson_r(x: Sequence[float], y: Sequence[float]) -> float:
    """Pearson r after filtering non-finite values."""
    x, y = clean_finite_pair(x, y)
    if len(x) <= 2:
        return np.nan
    return float(stats.pearsonr(x, y)[0])


def flatten_upper_triangle(matrix: pd.DataFrame, traits: Optional[Sequence[str]] = None) -> pd.Series:
    """Flatten the upper triangle of a symmetric trait-by-trait matrix."""
    if traits is not None:
        matrix = matrix.loc[list(traits), list(traits)]
    mask = np.triu(np.ones(matrix.shape, dtype=bool), k=1)
    return matrix.where(mask).stack()


def common_ordered_traits(*matrices: pd.DataFrame, preferred_order: Optional[Sequence[str]] = None) -> List[str]:
    """Trait intersection across matrices, preserving preferred_order when supplied."""
    common = set.intersection(*(set(m.index) for m in matrices))
    if preferred_order is not None:
        return [t for t in preferred_order if t in common]
    return sorted(common)


## Genetic correlation matrices


In [6]:
_RG_RE = re.compile(
    r"^Genetic Correlation:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*\(",
    re.MULTILINE,
)
_P_RE = re.compile(
    r"^P:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)",
    re.MULTILINE,
)


def parse_ldsc_log(file_path: Path) -> Tuple[float, float]:
    """Parse one LDSC genetic-correlation log and return (rg, p)."""
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    rg_match = _RG_RE.search(text)
    p_match = _P_RE.search(text)
    if rg_match is None or p_match is None:
        raise ValueError(f"Could not parse rg/p from {file_path}")
    return float(rg_match.group(1)), float(p_match.group(1))


def build_genetic_correlation_matrices(traits: Sequence[str], ldsc_cor_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build symmetric genetic-correlation and p-value matrices from LDSC logs.

    Expected filenames are either:
        cor.{trait1}.{trait2}.log
    or the reverse trait order.
    """
    traits = list(traits)
    rg_matrix = pd.DataFrame(np.nan, index=traits, columns=traits, dtype=float)
    pval_matrix = pd.DataFrame(np.nan, index=traits, columns=traits, dtype=float)

    for trait in traits:
        rg_matrix.loc[trait, trait] = 1.0
        pval_matrix.loc[trait, trait] = 0.0

    for trait1, trait2 in itertools.combinations(traits, 2):
        forward = ldsc_cor_dir / f"cor.{trait1}.{trait2}.log"
        reverse = ldsc_cor_dir / f"cor.{trait2}.{trait1}.log"
        log_path = forward if forward.exists() else reverse if reverse.exists() else None

        if log_path is None:
            warnings.warn(f"Missing LDSC log for pair ({trait1}, {trait2}).")
            continue

        try:
            rg, pval = parse_ldsc_log(log_path)
        except Exception as exc:
            warnings.warn(f"Failed to parse {log_path}: {exc}")
            continue

        rg_matrix.loc[trait1, trait2] = rg_matrix.loc[trait2, trait1] = rg
        pval_matrix.loc[trait1, trait2] = pval_matrix.loc[trait2, trait1] = pval

    return rg_matrix, pval_matrix


def drop_sparse_genetic_traits(
    genet_cor: pd.DataFrame,
    genet_cor_pval: pd.DataFrame,
    *,
    max_missing: int = 5,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Drop rows/columns with too many missing genetic correlations."""
    min_non_missing_cols = max(1, len(genet_cor.columns) - max_missing)
    kept_rows = genet_cor.dropna(thresh=min_non_missing_cols).index

    genet_cor = genet_cor.loc[kept_rows, kept_rows]
    genet_cor_pval = genet_cor_pval.loc[kept_rows, kept_rows]

    min_non_missing_rows = max(1, len(genet_cor) - max_missing)
    kept_cols = genet_cor.dropna(axis=1, thresh=min_non_missing_rows).columns

    return genet_cor.loc[kept_cols, kept_cols], genet_cor_pval.loc[kept_cols, kept_cols]


## Gene-level and pathway-level correlation matrices


In [7]:
def load_trait_gene_weights(gs_split_dir: Path, traits: Sequence[str]) -> Dict[str, Dict[str, float]]:
    """Read scDRS gene-set files into {trait: {gene: weight}}."""
    trait_gene_weights: Dict[str, Dict[str, float]] = {}

    for trait in traits:
        path = gs_split_dir / trait
        if not path.exists():
            warnings.warn(f"Missing gene-set split file for {trait}: {path}")
            continue

        lines = path.read_text().splitlines()
        if len(lines) < 2:
            warnings.warn(f"Gene-set split file has fewer than two lines: {path}")
            continue

        fields = lines[1].strip().split("\t")
        if len(fields) < 2:
            warnings.warn(f"Could not parse gene-set line in {path}")
            continue

        geneset = fields[1]
        weights = {}
        for pair in geneset.strip(",").split(","):
            if not pair:
                continue
            gene, weight = pair.split(":")
            weights[gene] = float(weight)

        trait_gene_weights[trait] = weights

    return trait_gene_weights


def build_pathway_gene_map(
    trait_gene_weights: Mapping[str, Mapping[str, float]],
    *,
    sources: Sequence[str] = ("GO:BP",),
    min_pathway_genes: int = 5,
    jaccard_prune_threshold: float = 0.30,
    cache_path: Optional[Path] = None,
) -> Dict[str, set]:
    """Map genes to enriched pathways, then prune redundant pathways by Jaccard overlap."""
    if cache_path is not None and cache_path.exists():
        enrichment_results = pd.read_pickle(cache_path)
    else:
        if GProfiler is None:
            raise ImportError("gprofiler is not installed. Install it or provide PATHWAY_ENRICHMENT_CACHE.")

        genes = sorted({gene for weights in trait_gene_weights.values() for gene in weights})
        gp = GProfiler(return_dataframe=True)
        enrichment_results = gp.profile(
            organism="hsapiens",
            query=genes,
            sources=list(sources),
            no_evidences=False,
        )
        if cache_path is not None:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            enrichment_results.to_pickle(cache_path)

    if "intersections" not in enrichment_results.columns or "name" not in enrichment_results.columns:
        raise ValueError("gProfiler results must contain 'name' and 'intersections' columns.")

    gene_to_pathways: Dict[str, set] = {}
    for _, row in enrichment_results.iterrows():
        pathway_name = row["name"]
        genes = row["intersections"]
        if not isinstance(genes, (list, tuple, set)):
            continue
        for gene in genes:
            gene_to_pathways.setdefault(gene, set()).add(pathway_name)

    pathway_to_genes: Dict[str, set] = {}
    for gene, pathways in gene_to_pathways.items():
        for pathway in pathways:
            pathway_to_genes.setdefault(pathway, set()).add(gene)

    pathway_to_genes = {
        pathway: genes
        for pathway, genes in pathway_to_genes.items()
        if len(genes) >= min_pathway_genes
    }

    remaining_pathways = set(pathway_to_genes)
    for pathway1, pathway2 in itertools.combinations(sorted(pathway_to_genes), 2):
        if pathway1 not in remaining_pathways or pathway2 not in remaining_pathways:
            continue

        genes1 = pathway_to_genes[pathway1]
        genes2 = pathway_to_genes[pathway2]
        union = genes1 | genes2
        jaccard = len(genes1 & genes2) / len(union) if union else 0.0

        if jaccard >= jaccard_prune_threshold:
            remove = pathway1 if len(genes1) < len(genes2) else pathway2
            remaining_pathways.remove(remove)

    pruned_gene_to_pathways: Dict[str, set] = {}
    for gene, pathways in gene_to_pathways.items():
        kept_pathways = {pathway for pathway in pathways if pathway in remaining_pathways}
        if kept_pathways:
            pruned_gene_to_pathways[gene] = kept_pathways

    return pruned_gene_to_pathways


def build_pathway_correlation(
    trait_gene_weights: Mapping[str, Mapping[str, float]],
    gene_to_pathways: Mapping[str, set],
) -> pd.DataFrame:
    """Build weighted trait-by-pathway matrix and return trait-trait pathway correlation."""
    traits = list(trait_gene_weights)
    pathways = sorted({pathway for pathways in gene_to_pathways.values() for pathway in pathways})

    trait_pathway_counts = pd.DataFrame(0.0, index=traits, columns=pathways)
    for trait, weights in trait_gene_weights.items():
        for gene, weight in weights.items():
            for pathway in gene_to_pathways.get(gene, []):
                trait_pathway_counts.loc[trait, pathway] += weight

    standardized = (trait_pathway_counts - trait_pathway_counts.mean(axis=0)) / trait_pathway_counts.std(axis=0)
    standardized = standardized.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return standardized.T.corr(method="pearson")


def build_gene_level_correlation(gene_zstat_file: Path, traits: Sequence[str]) -> pd.DataFrame:
    """Build trait-trait correlation from MAGMA gene-level Z-statistics."""
    gene_df = pd.read_csv(gene_zstat_file, sep="\t").dropna()
    available_traits = [trait for trait in traits if trait in gene_df.columns]
    if not available_traits:
        raise ValueError("No requested traits were found in the MAGMA gene-level file.")

    gene_df = gene_df[available_traits].astype(float)
    standardized = (gene_df - gene_df.mean(axis=0)) / gene_df.std(axis=0)
    standardized = standardized.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how="any")
    return standardized.corr(method="pearson")


## scDRS+ score matrices and functional-correlation p-values


In [8]:
def score_file(score_dir: Path, trait: str, suffix: str) -> Path:
    """Construct a scDRS score-file path."""
    return score_dir / f"{trait}.{suffix}"


def traits_with_score_file(score_dir: Path, traits: Sequence[str], suffix: str) -> List[str]:
    """Filter traits to those with a requested score file."""
    return [trait for trait in traits if score_file(score_dir, trait, suffix).exists()]


def build_score_matrix(
    score_dir: Path,
    traits: Sequence[str],
    *,
    suffix: str,
    score_col: str = "norm_score",
) -> pd.DataFrame:
    """Read one score column for each trait and combine into a cell-by-trait matrix."""
    scores = {}
    missing = []

    for trait in traits:
        path = score_file(score_dir, trait, suffix)
        if not path.exists():
            missing.append(trait)
            continue
        df = pd.read_csv(path, sep="\t", index_col=0)
        if score_col not in df.columns:
            raise ValueError(f"{path} is missing required column {score_col!r}")
        scores[trait] = df[score_col]

    if missing:
        warnings.warn(f"Skipped {len(missing)} traits without {suffix}: {missing[:10]}")
    if not scores:
        raise ValueError(f"No score files found in {score_dir} for suffix {suffix!r}")

    return pd.DataFrame(scores)


def compute_conditional_correlation_with_mc_pvals(
    score_dir: Path,
    traits: Sequence[str],
    *,
    n_ctrl: int = 1000,
    score_col: str = "norm_score",
) -> pd.DataFrame:
    """
    Compute trait-pair functional correlations and Monte Carlo p-values.

    For pair (trait1, trait2), the null distribution is generated by correlating
    trait1's observed scores with trait2's control scores.
    """
    traits = [trait for trait in traits if score_file(score_dir, trait, "conditional.tagging_score.gz").exists()]
    trait_scores: Dict[str, pd.Series] = {}
    ctrl_scores: Dict[str, np.ndarray] = {}

    for trait in traits:
        path = score_file(score_dir, trait, "conditional.tagging_score.gz")
        df = pd.read_csv(path, sep="\t", index_col=0)
        if score_col not in df.columns:
            raise ValueError(f"{path} is missing required column {score_col!r}")

        ctrl_cols = [f"ctrl_norm_score_{i}" for i in range(n_ctrl) if f"ctrl_norm_score_{i}" in df.columns]
        if not ctrl_cols:
            raise ValueError(f"{path} does not contain any ctrl_norm_score_* columns.")

        trait_scores[trait] = df[score_col]
        ctrl_scores[trait] = df[ctrl_cols].to_numpy(dtype=float)

    score_matrix = pd.DataFrame(trait_scores)
    results = []

    for trait1 in traits:
        trait1_scores = score_matrix[trait1].to_numpy(dtype=float)
        trait1_mean = np.nanmean(trait1_scores)
        trait1_std = np.nanstd(trait1_scores)

        for trait2 in traits:
            trait2_scores = score_matrix[trait2].to_numpy(dtype=float)
            real_corr = pearson_r(trait1_scores, trait2_scores)

            ctrl_array = ctrl_scores[trait2]
            ctrl_means = np.nanmean(ctrl_array, axis=0)
            ctrl_stds = np.nanstd(ctrl_array, axis=0)
            covariance = np.nanmean((ctrl_array - ctrl_means) * (trait1_scores[:, None] - trait1_mean), axis=0)

            with np.errstate(divide="ignore", invalid="ignore"):
                ctrl_corrs = covariance / (ctrl_stds * trait1_std)

            ctrl_corrs = ctrl_corrs[np.isfinite(ctrl_corrs)]
            if len(ctrl_corrs) == 0 or not np.isfinite(real_corr):
                mc_pval = np.nan
            else:
                mc_pval = (1 + np.sum(ctrl_corrs >= real_corr)) / (len(ctrl_corrs) + 1)

            results.append({
                "trait1": trait1,
                "trait2": trait2,
                "correlation": real_corr,
                "mc_pval": mc_pval,
            })

    return pd.DataFrame(results, columns=["trait1", "trait2", "correlation", "mc_pval"])


## Bootstrap summaries


In [9]:
def summarize_distribution(values: Sequence[float]) -> Dict[str, float]:
    """Mean and percentile confidence interval for a bootstrap distribution."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {"mean": np.nan, "ci_lower": np.nan, "ci_upper": np.nan, "n": 0}
    return {
        "mean": float(np.mean(values)),
        "ci_lower": float(np.percentile(values, 2.5)),
        "ci_upper": float(np.percentile(values, 97.5)),
        "n": int(len(values)),
    }


def transformed_upper_triangle(matrix: pd.DataFrame, traits: Sequence[str], transform: Optional[str] = None) -> pd.Series:
    """Upper-triangle values, optionally transformed."""
    values = flatten_upper_triangle(matrix, traits=traits)
    if transform == "abs":
        values = values.abs()
    return values


def bootstrap_pearson_distributions(
    target_matrix: pd.DataFrame,
    predictor_matrices: Mapping[str, pd.DataFrame],
    *,
    preferred_traits: Optional[Sequence[str]] = None,
    n_iterations: int = 1000,
    sample_frac: float = 0.8,
    random_state: int = 0,
) -> Dict[str, object]:
    """Bootstrap Pearson correlations between predictor matrices and a target matrix."""
    all_matrices = [target_matrix] + list(predictor_matrices.values())
    traits = common_ordered_traits(*all_matrices, preferred_order=preferred_traits)
    if len(traits) < 3:
        raise ValueError(f"Need at least 3 common traits; found {len(traits)}.")

    transforms = {"abs_genetic": "abs"}
    y_full = transformed_upper_triangle(target_matrix, traits)

    baseline = {}
    for name, matrix in predictor_matrices.items():
        x_full = transformed_upper_triangle(matrix, traits, transform=transforms.get(name))
        baseline[name] = pearson_r(x_full, y_full)

    rng = np.random.default_rng(random_state)
    n_sampled = max(3, int(len(traits) * sample_frac))
    n_sampled = min(n_sampled, len(traits))
    boot = {name: [] for name in predictor_matrices}

    for _ in range(n_iterations):
        sampled_traits = rng.choice(traits, size=n_sampled, replace=False).tolist()
        y = transformed_upper_triangle(target_matrix, sampled_traits)
        for name, matrix in predictor_matrices.items():
            x = transformed_upper_triangle(matrix, sampled_traits, transform=transforms.get(name))
            boot[name].append(pearson_r(x, y))

    boot = {
        name: [float(v) for v in values if np.isfinite(v)]
        for name, values in boot.items()
    }

    return {
        "traits": traits,
        "n_sampled": n_sampled,
        "baseline": baseline,
        "boot": boot,
        "summary": {name: summarize_distribution(values) for name, values in boot.items()},
    }


def min_traits_for_partial_r2(n_predictors: int) -> int:
    """Smallest trait count with enough pairwise observations for OLS."""
    n_traits = 2
    while (n_traits * (n_traits - 1)) // 2 <= n_predictors + 1:
        n_traits += 1
    return n_traits


def partial_r2_for_subset(
    target_matrix: pd.DataFrame,
    predictor_matrices: Mapping[str, pd.DataFrame],
    traits: Sequence[str],
) -> Optional[Dict[str, float]]:
    """Drop-one partial R² values for predictor matrices against a target matrix."""
    transforms = {"abs_genetic": "abs"}
    y = transformed_upper_triangle(target_matrix, traits).rename("target")

    data = {"target": y.reset_index(drop=True)}
    for name, matrix in predictor_matrices.items():
        x = transformed_upper_triangle(matrix, traits, transform=transforms.get(name))
        data[name] = x.reset_index(drop=True)

    df = pd.DataFrame(data).replace([np.inf, -np.inf], np.nan).dropna()
    predictors = list(predictor_matrices)
    if df.shape[0] <= len(predictors) + 1:
        return None

    y_values = df["target"]
    x_full = sm.add_constant(df[predictors])
    full_model = sm.OLS(y_values, x_full).fit()
    full_r2 = full_model.rsquared

    partial = {}
    for predictor in predictors:
        reduced = x_full.drop(columns=[predictor])
        reduced_r2 = sm.OLS(y_values, reduced).fit().rsquared
        partial[predictor] = float(full_r2 - reduced_r2)

    return partial


def bootstrap_partial_r2_distributions(
    target_matrix: pd.DataFrame,
    predictor_matrices: Mapping[str, pd.DataFrame],
    *,
    preferred_traits: Optional[Sequence[str]] = None,
    n_iterations: int = 1000,
    sample_frac: float = 0.8,
    random_state: int = 0,
) -> Dict[str, object]:
    """Bootstrap drop-one partial R² distributions."""
    all_matrices = [target_matrix] + list(predictor_matrices.values())
    traits = common_ordered_traits(*all_matrices, preferred_order=preferred_traits)
    min_traits = min_traits_for_partial_r2(len(predictor_matrices))
    if len(traits) < min_traits:
        raise ValueError(f"Need at least {min_traits} common traits; found {len(traits)}.")

    baseline = partial_r2_for_subset(target_matrix, predictor_matrices, traits)
    if baseline is None:
        raise ValueError("Could not fit baseline partial R² model.")

    rng = np.random.default_rng(random_state)
    n_sampled = max(min_traits, int(len(traits) * sample_frac))
    n_sampled = min(n_sampled, len(traits))
    boot = {name: [] for name in predictor_matrices}

    for _ in range(n_iterations):
        sampled_traits = rng.choice(traits, size=n_sampled, replace=False).tolist()
        partial = partial_r2_for_subset(target_matrix, predictor_matrices, sampled_traits)
        if partial is None:
            continue
        for name in predictor_matrices:
            boot[name].append(partial.get(name, np.nan))

    boot = {
        name: [float(v) for v in values if np.isfinite(v)]
        for name, values in boot.items()
    }

    return {
        "traits": traits,
        "n_sampled": n_sampled,
        "baseline": baseline,
        "boot": boot,
        "summary": {name: summarize_distribution(values) for name, values in boot.items()},
    }


## Plot helpers


In [10]:
def add_top_lines(
    ax,
    color_counts: Sequence[Tuple[str, int]],
    names: Sequence[str],
    *,
    y_axes: float = 1.01,
    text_offset: float = 0.02,
    linewidth: float = 4,
    fontsize: float = 32,
    fontweight: str = "bold",
) -> None:
    """Add colored group bars and labels above a heatmap."""
    total = max(1, sum(count for _, count in color_counts))
    start = 0
    for (color, count), name in zip(color_counts, names):
        end = start + int(count)
        xmin, xmax = start / total, end / total
        ax.add_line(Line2D(
            [xmin, xmax], [y_axes, y_axes], transform=ax.transAxes,
            color=color, linewidth=linewidth, solid_capstyle="butt", clip_on=False,
        ))
        if xmax > xmin:
            ax.text(
                (xmin + xmax) / 2, y_axes + text_offset, name,
                ha="center", va="bottom", fontsize=fontsize,
                color=color, fontweight=fontweight, transform=ax.transAxes,
                clip_on=False,
            )
        start = end


def add_right_lines(
    ax,
    color_counts: Sequence[Tuple[str, int]],
    names: Sequence[str],
    *,
    x_axes: float = 1.01,
    text_offset: float = 0.02,
    linewidth: float = 4,
    fontsize: float = 32,
    fontweight: str = "bold",
) -> None:
    """Add colored group bars and labels to the right side of a heatmap."""
    total = max(1, sum(count for _, count in color_counts))
    start = 0
    for (color, count), name in zip(color_counts, names):
        end = start + int(count)
        ymin, ymax = start / total, end / total
        ax.add_line(Line2D(
            [x_axes, x_axes], [ymin, ymax], transform=ax.transAxes,
            color=color, linewidth=linewidth, solid_capstyle="butt", clip_on=False,
        ))
        if ymax > ymin:
            ax.text(
                x_axes + text_offset, (ymin + ymax) / 2, name,
                ha="left", va="center", fontsize=fontsize,
                color=color, fontweight=fontweight, transform=ax.transAxes,
                clip_on=False, rotation=-90,
            )
        start = end


def trait_group_color(trait: str, subset_traits: Sequence[str]) -> str:
    """Color-code a curated subset trait by its position."""
    idx = list(subset_traits).index(trait)
    if idx < 4:
        return "red"
    if idx < 8:
        return "blue"
    return "green"


In [11]:
def plot_split_functional_genetic_heatmap(
    *,
    correlation_df: pd.DataFrame,
    genet_cor: pd.DataFrame,
    genet_cor_pval: pd.DataFrame,
    subset_traits: Sequence[str],
    trait_name_dict: Mapping[str, str],
    dataset: str,
    out_path: Path,
    group_spec: Sequence[Tuple[str, str, int]] = SUBSET_GROUPS,
    functional_alpha: float = 0.05,
    genetic_alpha: float = 0.05,
) -> None:
    """
    Plot one split matrix:
      - lower-left section: functional correlation
      - upper-right section: genetic correlation
      - diagonal: white separation gap
    """
    available_subset = [t for t in subset_traits if t in set(correlation_df["trait1"]).union(correlation_df["trait2"])]
    if len(available_subset) < 3:
        warnings.warn(f"Skipping split heatmap for {dataset}: fewer than three subset traits available.")
        return

    cor_matrix = correlation_df.pivot(index="trait1", columns="trait2", values="correlation")
    pval_matrix = correlation_df.pivot(index="trait1", columns="trait2", values="mc_pval")

    all_traits = sorted(set(available_subset) | set(cor_matrix.index) | set(cor_matrix.columns) | set(genet_cor.index) | set(genet_cor.columns))
    cor_matrix = cor_matrix.reindex(index=all_traits, columns=all_traits)
    pval_matrix = pval_matrix.reindex(index=all_traits, columns=all_traits)

    functional_sig = pd.DataFrame(False, index=all_traits, columns=all_traits)
    for trait1 in all_traits:
        others = [trait for trait in all_traits if trait != trait1]
        pvals = pval_matrix.loc[trait1, others].dropna()
        if not pvals.empty:
            _, corrected, _, _ = multipletests(pvals.values, alpha=functional_alpha, method="bonferroni")
            functional_sig.loc[trait1, pvals.index] = corrected < functional_alpha
    functional_sig = functional_sig & functional_sig.T

    genet = genet_cor.reindex(index=all_traits, columns=all_traits)
    genet_p = genet_cor_pval.reindex(index=all_traits, columns=all_traits)
    genetic_sig = pd.DataFrame(False, index=all_traits, columns=all_traits)
    for trait1 in all_traits:
        others = [trait for trait in all_traits if trait != trait1]
        pvals = genet_p.loc[trait1, others].dropna()
        if not pvals.empty:
            _, corrected, _, _ = multipletests(pvals.values, alpha=genetic_alpha, method="fdr_bh")
            genetic_sig.loc[trait1, pvals.index] = corrected < genetic_alpha
    genetic_sig = genetic_sig & genetic_sig.T

    trait_labels = {trait: trait_name_dict.get(trait, trait) for trait in all_traits}
    ordered_labels = [trait_labels[trait] for trait in available_subset]

    cor_named = cor_matrix.rename(index=trait_labels, columns=trait_labels).loc[ordered_labels, ordered_labels]
    functional_sig_named = functional_sig.rename(index=trait_labels, columns=trait_labels).loc[ordered_labels, ordered_labels]
    genet_named = genet.rename(index=trait_labels, columns=trait_labels).loc[ordered_labels, ordered_labels]
    genetic_sig_named = genetic_sig.rename(index=trait_labels, columns=trait_labels).loc[ordered_labels, ordered_labels]

    n = len(ordered_labels)
    combined = pd.DataFrame(np.nan, index=ordered_labels, columns=ordered_labels)
    for i in range(n):
        for j in range(n):
            if i > j:
                combined.iat[i, j] = cor_named.iat[i, j]
            elif i < j:
                combined.iat[i, j] = genet_named.iat[i, j]

    fig, ax = plt.subplots(figsize=(22, 18))
    sns.heatmap(
        combined,
        cmap="vlag",
        linewidths=0.5,
        linecolor="lightgray",
        vmin=-1,
        vmax=1,
        square=True,
        cbar=True,
        ax=ax,
        cbar_kws={"label": "Correlation", "shrink": 0.70, "ticks": [-1, 0, 1]},
    )

    # Plot-level diagonal separation between the lower-left and upper-right sections.
    for i in range(n):
        ax.add_patch(Rectangle((i, i), 1, 1, facecolor="white", edgecolor="white", linewidth=0, zorder=7))
    ax.plot([0, n], [0, n], color="white", linewidth=18, solid_capstyle="butt", zorder=8, clip_on=False)
    ax.plot([0, n], [0, n], color="lightgray", linewidth=1.2, zorder=9, clip_on=False)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=34)
    cbar.set_label("Correlation", size=36)

    ax.set_xticks(np.arange(n) + 0.5)
    ax.set_yticks(np.arange(n) + 0.5)
    ax.set_xticklabels(ordered_labels, rotation=45, fontsize=22, ha="right")
    ax.set_yticklabels(ordered_labels, rotation=0, fontsize=22)
    ax.set_xlabel("")
    ax.set_ylabel("")

    for idx, trait in enumerate(available_subset):
        color = trait_group_color(trait, subset_traits)
        ax.get_xticklabels()[idx].set_color(color)
        ax.get_yticklabels()[idx].set_color(color)

    # Group bars and dashed boxes. Counts are adapted if some subset traits are unavailable.
    group_counts = []
    cursor = 0
    available_set = set(available_subset)
    for name, color, count in group_spec:
        group_traits = list(subset_traits)[cursor:cursor + count]
        actual = sum(trait in available_set for trait in group_traits)
        group_counts.append((color, actual))
        cursor += count

    group_names = [name for name, _, _ in group_spec]
    add_top_lines(ax, group_counts, group_names, fontsize=30)
    add_right_lines(ax, list(reversed(group_counts)), list(reversed(group_names)), fontsize=30)

    start = 0
    for _, size in group_counts:
        if size > 0:
            ax.add_patch(Rectangle((start, start), size, size, fill=False, edgecolor="black", linestyle="--", linewidth=1.5, zorder=25))
        start += size

    for i, label1 in enumerate(ordered_labels):
        for j, label2 in enumerate(ordered_labels):
            if i == j:
                continue
            if i > j and bool(functional_sig_named.loc[label1, label2]):
                ax.text(
                    j + 0.5,
                    i + 0.53,
                    "★",
                    ha="center",
                    va="center",
                    color="gold",
                    fontsize=36,
                    fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2.8, foreground="black")],
                    zorder=30,
                )
            if i < j and bool(genetic_sig_named.loc[label1, label2]):
                ax.text(j + 0.5, i + 0.53, "★", ha="center", va="center", color="black", fontsize=36, fontweight="bold", zorder=30)

    legend_elements = [
        Line2D([], [], color="gold", marker="*", linestyle="None", markersize=26, markeredgecolor="black", markeredgewidth=1.8, label="Sig. functional correlation"),
        Line2D([], [], color="black", marker="*", linestyle="None", markersize=26, markeredgecolor="black", label="Sig. genetic correlation"),
    ]
    fig.legend(handles=legend_elements, loc="upper right", bbox_to_anchor=(0.90, 1.05), fontsize=26)

    ax.text(0.02, 0.05, "Functional correlation", transform=ax.transAxes, ha="left", va="bottom", fontsize=28, fontweight="bold")
    ax.text(0.98, 0.95, "Genetic correlation", transform=ax.transAxes, ha="right", va="top", fontsize=28, fontweight="bold")

    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()


In [12]:
def plot_bootstrap_summary(
    *,
    corr_functional: Mapping[str, object],
    pr2_functional: Mapping[str, object],
    corr_marginal: Mapping[str, object],
    pr2_marginal: Mapping[str, object],
    dataset: str,
    out_path: Path,
) -> None:
    """Plot Pearson-r and partial-R² bootstrap summaries for functional and marginal targets."""
    order = ["genetic", "abs_genetic", "gene", "pathway", "marginal"]

    def values_with_ci(result: Mapping[str, object], key_order: Sequence[str], as_percent: bool = False) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        means, lows, highs = [], [], []
        summary = result["summary"]
        for key in key_order:
            item = summary.get(key, {"mean": np.nan, "ci_lower": np.nan, "ci_upper": np.nan})
            scale = 100.0 if as_percent else 1.0
            means.append(item["mean"] * scale)
            lows.append(item["ci_lower"] * scale)
            highs.append(item["ci_upper"] * scale)
        return np.array(means), np.array(lows), np.array(highs)

    fig, axes = plt.subplots(2, 1, figsize=(8.5, 12), sharex=True)
    x = np.arange(len(order))
    width = 0.38
    labels = [COMPARATOR_LABELS.get(k, k) for k in order]

    panels = [
        (axes[0], corr_functional, corr_marginal, "Pearson's r", False),
        (axes[1], pr2_functional, pr2_marginal, "Partial variance explained (%)", True),
    ]

    for ax, functional_result, marginal_result, ylabel, as_percent in panels:
        f_mean, f_low, f_high = values_with_ci(functional_result, order, as_percent=as_percent)
        m_mean, m_low, m_high = values_with_ci(marginal_result, order, as_percent=as_percent)

        ax.bar(x - width / 2, f_mean, width=width, color="tab:blue", label="Functional correlation")
        ax.errorbar(x - width / 2, f_mean, yerr=np.vstack([f_mean - f_low, f_high - f_mean]), fmt="none", ecolor="black", capsize=5, linewidth=2)

        ax.bar(x + width / 2, m_mean, width=width, color="tab:red", label="Marginal functional correlation")
        ax.errorbar(x + width / 2, m_mean, yerr=np.vstack([m_mean - m_low, m_high - m_mean]), fmt="none", ecolor="black", capsize=5, linewidth=2)

        ax.set_ylabel(ylabel, fontsize=16)
        ax.tick_params(axis="y", labelsize=13)
        ax.axhline(0, color="black", linewidth=0.8)

    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=14)
    axes[0].legend(frameon=True, fontsize=12)
    fig.suptitle(f"Correlation-analysis summaries: {dataset}", fontsize=18)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()


In [13]:
def plot_functional_comparator_scatter_grid(
    *,
    functional_correlation: pd.DataFrame,
    genet_cor: pd.DataFrame,
    gene_level_correlation: pd.DataFrame,
    pathway_correlation: pd.DataFrame,
    marginal_correlation: pd.DataFrame,
    dataset: str,
    out_path: Path,
    preferred_traits: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    """
    Make the requested five-panel scatter figure:
      functional correlation vs genetic, |genetic|, gene-level, pathway, and marginal correlations.
    """
    comparators = {
        "genetic": genet_cor,
        "abs_genetic": genet_cor,
        "gene": gene_level_correlation,
        "pathway": pathway_correlation,
        "marginal": marginal_correlation,
    }
    traits = common_ordered_traits(functional_correlation, *comparators.values(), preferred_order=preferred_traits)
    y = flatten_upper_triangle(functional_correlation, traits=traits)

    fig, axes = plt.subplots(1, 5, figsize=(25, 5), sharey=True)
    rows = []

    for ax, (name, matrix) in zip(axes, comparators.items()):
        x = flatten_upper_triangle(matrix, traits=traits)
        if name == "abs_genetic":
            x = x.abs()

        x_clean, y_clean = clean_finite_pair(x, y)
        r, p = (np.nan, np.nan)
        if len(x_clean) > 2:
            r, p = stats.pearsonr(x_clean, y_clean)

        rows.append({
            "dataset": dataset,
            "comparator": name,
            "n_pairs": int(len(x_clean)),
            "pearson_r": float(r) if np.isfinite(r) else np.nan,
            "pearson_p": float(p) if np.isfinite(p) else np.nan,
        })

        ax.scatter(x_clean, y_clean, alpha=0.65, s=20)
        if len(x_clean) > 2 and np.nanstd(x_clean) > 0:
            slope, intercept = np.polyfit(x_clean, y_clean, deg=1)
            xx = np.linspace(np.nanmin(x_clean), np.nanmax(x_clean), 100)
            ax.plot(xx, intercept + slope * xx, linestyle="--", linewidth=1.5)

        ax.set_title(f"{COMPARATOR_LABELS[name]}\nr = {r:.3f}", fontsize=16)
        ax.set_xlabel(f"{COMPARATOR_LABELS[name]} correlation", fontsize=13)
        ax.tick_params(axis="both", labelsize=11)

    axes[0].set_ylabel("Functional correlation", fontsize=14)
    fig.suptitle(f"Functional correlation vs comparator correlations: {dataset}", fontsize=18)
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    return pd.DataFrame(rows)


In [14]:
def plot_cross_dataset_functional_correlations(
    dataset_results: Mapping[str, Mapping[str, object]],
    *,
    reference_dataset: str = "tms_facs",
    out_path: Path,
) -> pd.DataFrame:
    """Compare functional-correlation matrices across datasets."""
    reference = dataset_results[reference_dataset]["functional_correlation"]
    rows = []
    comparisons = [name for name in dataset_results if name != reference_dataset]

    fig, axes = plt.subplots(1, len(comparisons), figsize=(6 * len(comparisons), 5), constrained_layout=True)
    if len(comparisons) == 1:
        axes = [axes]

    for ax, dataset in zip(axes, comparisons):
        other = dataset_results[dataset]["functional_correlation"]
        traits = common_ordered_traits(reference, other)
        x = flatten_upper_triangle(reference, traits=traits)
        y = flatten_upper_triangle(other, traits=traits)
        x_clean, y_clean = clean_finite_pair(x, y)

        if len(x_clean) > 2:
            r, p = stats.pearsonr(x_clean, y_clean)
        else:
            r, p = np.nan, np.nan

        rows.append({
            "reference_dataset": reference_dataset,
            "comparison_dataset": dataset,
            "n_traits": int(len(traits)),
            "n_pairs": int(len(x_clean)),
            "pearson_r": float(r) if np.isfinite(r) else np.nan,
            "pearson_p": float(p) if np.isfinite(p) else np.nan,
        })

        ax.scatter(x_clean, y_clean, alpha=0.70)
        if len(x_clean) > 0:
            lo = np.nanmin([x_clean.min(), y_clean.min()])
            hi = np.nanmax([x_clean.max(), y_clean.max()])
            ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
        ax.set_xlabel(f"{reference_dataset} functional correlation", fontsize=13)
        ax.set_ylabel(f"{dataset} functional correlation", fontsize=13)
        ax.set_title(f"{reference_dataset} vs {dataset}\nr = {r:.3f}", fontsize=16)

    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    return pd.DataFrame(rows)


## Dataset-level analysis function


In [15]:
def run_dataset_analysis(
    *,
    dataset_config: DatasetConfig,
    traits: Sequence[str],
    genet_cor: pd.DataFrame,
    genet_cor_pval: pd.DataFrame,
    gene_level_correlation: pd.DataFrame,
    pathway_correlation: pd.DataFrame,
    subset_traits: Sequence[str],
    output_root: Path,
) -> Dict[str, object]:
    """Run the full cleaned analysis for one dataset/folder pair."""
    dataset = dataset_config.name
    score_dir = dataset_config.score_dir
    out_dir = ensure_output_dir(output_root / dataset)

    conditional_traits = traits_with_score_file(score_dir, traits, "conditional.tagging_score.gz")
    marginal_traits = traits_with_score_file(score_dir, traits, "marginal_score.gz")
    available_traits = [
        trait for trait in traits
        if trait in conditional_traits
        and trait in marginal_traits
        and trait in genet_cor.index
        and trait in gene_level_correlation.index
        and trait in pathway_correlation.index
    ]

    if len(available_traits) < 3:
        raise ValueError(f"{dataset}: fewer than three traits are available across all matrices.")

    print(f"\n=== {dataset} ===")
    print(f"Score folder: {score_dir}")
    print(f"Traits used for matrix analyses: {len(available_traits)}")

    conditional_scores = build_score_matrix(
        score_dir,
        available_traits,
        suffix="conditional.tagging_score.gz",
        score_col="norm_score",
    )
    marginal_scores = build_score_matrix(
        score_dir,
        available_traits,
        suffix="marginal_score.gz",
        score_col="norm_score",
    )

    functional_correlation = conditional_scores.corr(method="pearson")
    marginal_correlation = marginal_scores.corr(method="pearson")

    # Save every correlation matrix used by this dataset in wide CSV form.
    dataset_matrices = {
        "functional_correlation": functional_correlation,
        "marginal_correlation": marginal_correlation,
        "genetic_correlation": genet_cor.reindex(index=available_traits, columns=available_traits),
        "genetic_correlation_pvalues": genet_cor_pval.reindex(index=available_traits, columns=available_traits),
        "gene_level_correlation": gene_level_correlation.reindex(index=available_traits, columns=available_traits),
        "pathway_correlation": pathway_correlation.reindex(index=available_traits, columns=available_traits),
    }
    for matrix_name, matrix in dataset_matrices.items():
        export_matrix_csv(matrix, out_dir / f"{dataset}.{matrix_name}.csv")

    functional_predictors = {
        "genetic": genet_cor,
        "abs_genetic": genet_cor,
        "gene": gene_level_correlation,
        "pathway": pathway_correlation,
        "marginal": marginal_correlation,
    }
    marginal_predictors = {
        "genetic": genet_cor,
        "abs_genetic": genet_cor,
        "gene": gene_level_correlation,
        "pathway": pathway_correlation,
    }

    corr_functional = bootstrap_pearson_distributions(
        functional_correlation,
        functional_predictors,
        preferred_traits=available_traits,
        n_iterations=N_BOOTSTRAP_ITERATIONS,
        sample_frac=BOOTSTRAP_SAMPLE_FRAC,
        random_state=RANDOM_SEED,
    )
    pr2_functional = bootstrap_partial_r2_distributions(
        functional_correlation,
        functional_predictors,
        preferred_traits=available_traits,
        n_iterations=N_BOOTSTRAP_ITERATIONS,
        sample_frac=BOOTSTRAP_SAMPLE_FRAC,
        random_state=RANDOM_SEED,
    )
    corr_marginal = bootstrap_pearson_distributions(
        marginal_correlation,
        marginal_predictors,
        preferred_traits=available_traits,
        n_iterations=N_BOOTSTRAP_ITERATIONS,
        sample_frac=BOOTSTRAP_SAMPLE_FRAC,
        random_state=RANDOM_SEED,
    )
    pr2_marginal = bootstrap_partial_r2_distributions(
        marginal_correlation,
        marginal_predictors,
        preferred_traits=available_traits,
        n_iterations=N_BOOTSTRAP_ITERATIONS,
        sample_frac=BOOTSTRAP_SAMPLE_FRAC,
        random_state=RANDOM_SEED,
    )

    summary_tables = []
    for target_name, result in [("functional", corr_functional), ("marginal", corr_marginal)]:
        for comparator, values in result["summary"].items():
            summary_tables.append({
                "dataset": dataset,
                "target": target_name,
                "metric": "pearson_r",
                "comparator": comparator,
                "baseline": result["baseline"].get(comparator, np.nan),
                "n_traits": len(result["traits"]),
                "n_sampled_traits": result["n_sampled"],
                "bootstrap_iterations_requested": N_BOOTSTRAP_ITERATIONS,
                **values,
            })
    for target_name, result in [("functional", pr2_functional), ("marginal", pr2_marginal)]:
        for comparator, values in result["summary"].items():
            summary_tables.append({
                "dataset": dataset,
                "target": target_name,
                "metric": "partial_r2",
                "comparator": comparator,
                "baseline": result["baseline"].get(comparator, np.nan),
                "n_traits": len(result["traits"]),
                "n_sampled_traits": result["n_sampled"],
                "bootstrap_iterations_requested": N_BOOTSTRAP_ITERATIONS,
                **values,
            })
    bootstrap_summary = pd.DataFrame(summary_tables)
    export_dataframe_csv(
        bootstrap_summary,
        out_dir / f"{dataset}.bootstrap_summary.csv",
        index=False,
    )

    plot_bootstrap_summary(
        corr_functional=corr_functional,
        pr2_functional=pr2_functional,
        corr_marginal=corr_marginal,
        pr2_marginal=pr2_marginal,
        dataset=dataset,
        out_path=out_dir / f"{dataset}.bootstrap_summary.png",
    )

    scatter_summary = plot_functional_comparator_scatter_grid(
        functional_correlation=functional_correlation,
        genet_cor=genet_cor,
        gene_level_correlation=gene_level_correlation,
        pathway_correlation=pathway_correlation,
        marginal_correlation=marginal_correlation,
        dataset=dataset,
        out_path=out_dir / f"{dataset}.functional_vs_comparators_5panel.png",
        preferred_traits=available_traits,
    )
    export_dataframe_csv(
        scatter_summary,
        out_dir / f"{dataset}.functional_vs_comparators_5panel.csv",
        index=False,
    )

    subset_available = [trait for trait in subset_traits if trait in available_traits]
    correlation_df = compute_conditional_correlation_with_mc_pvals(
        score_dir,
        subset_available,
        n_ctrl=FUNCTIONAL_MC_N_CTRL,
    )
    correlation_df.insert(0, "dataset", dataset)
    export_dataframe_csv(
        correlation_df,
        out_dir / f"{dataset}.subset_functional_correlation_mc_pvals.csv",
        index=False,
    )

    subset_functional_correlation = correlation_df.pivot(
        index="trait1",
        columns="trait2",
        values="correlation",
    ).reindex(index=subset_available, columns=subset_available)
    subset_functional_mc_pvalues = correlation_df.pivot(
        index="trait1",
        columns="trait2",
        values="mc_pval",
    ).reindex(index=subset_available, columns=subset_available)

    export_matrix_csv(
        subset_functional_correlation,
        out_dir / f"{dataset}.subset_functional_correlation_matrix.csv",
    )
    export_matrix_csv(
        subset_functional_mc_pvalues,
        out_dir / f"{dataset}.subset_functional_mc_pvalue_matrix.csv",
    )

    plot_split_functional_genetic_heatmap(
        correlation_df=correlation_df,
        genet_cor=genet_cor,
        genet_cor_pval=genet_cor_pval,
        subset_traits=subset_available,
        trait_name_dict=TRAIT_NAME_DICT,
        dataset=dataset,
        out_path=out_dir / f"{dataset}.functional_genetic_split_heatmap.png",
        functional_alpha=FUNCTIONAL_STAR_ALPHA,
        genetic_alpha=GENETIC_STAR_ALPHA,
    )

    return {
        "dataset": dataset,
        "score_dir": score_dir,
        "out_dir": out_dir,
        "traits": available_traits,
        "functional_correlation": functional_correlation,
        "marginal_correlation": marginal_correlation,
        "bootstrap_summary": bootstrap_summary,
        "scatter_summary": scatter_summary,
        "subset_correlation_df": correlation_df,
        "subset_functional_correlation": subset_functional_correlation,
        "subset_functional_mc_pvalues": subset_functional_mc_pvalues,
    }


## Build global comparator matrices


In [16]:
# Discover traits from the scDRS gene-set split files.
ALL_TRAITS = list_trait_files(GS_SPLIT_DIR, TRAITS_TO_DROP)
print(f"Discovered {len(ALL_TRAITS)} traits from {GS_SPLIT_DIR}")

# Genetic correlations from LDSC logs.
# LDSC per-pair logs are not in the public release; load the provided rg matrix
# instead and set p-values to NaN (no genetic significance stars). Everything
# else (sparse-trait filtering, plotting) is unchanged.
_rg_raw = pd.read_csv(GENET_COR_CSV, index_col=0)
genet_cor = _rg_raw.reindex(index=ALL_TRAITS, columns=ALL_TRAITS)
genet_cor_pval = pd.DataFrame(np.nan, index=ALL_TRAITS, columns=ALL_TRAITS)
genet_cor, genet_cor_pval = drop_sparse_genetic_traits(genet_cor, genet_cor_pval, max_missing=5)
print(f"Genetic correlation matrix shape after sparse-trait filtering: {genet_cor.shape}")

# Use the filtered genetic-correlation trait order for downstream global matrices.
TRAITS = [trait for trait in ALL_TRAITS if trait in genet_cor.index]

# Global wide-form CSV matrices and the retained trait order.
export_matrix_csv(genet_cor, GLOBAL_MATRIX_DIR / "genetic_correlation.csv")
export_matrix_csv(genet_cor_pval, GLOBAL_MATRIX_DIR / "genetic_correlation_pvalues.csv")
export_dataframe_csv(
    pd.DataFrame({"trait": TRAITS}),
    GLOBAL_MATRIX_DIR / "traits.csv",
    index=False,
)


Discovered 74 traits from /mnt/shared-workspace/scdrsfm/data/gene_sets/gs_split


Genetic correlation matrix shape after sparse-trait filtering: (73, 73)


{'csv': PosixPath('correlation_analysis_outputs/global_matrices/traits.csv'),
 'tsv': PosixPath('correlation_analysis_outputs/global_matrices/traits.tsv')}

In [17]:
# Pathway-level correlation.
trait_gene_weights = load_trait_gene_weights(GS_SPLIT_DIR, TRAITS)
gene_to_pathways = build_pathway_gene_map(
    trait_gene_weights,
    sources=PATHWAY_SOURCES,
    min_pathway_genes=MIN_PATHWAY_GENES,
    jaccard_prune_threshold=PATHWAY_JACCARD_PRUNE_THRESHOLD,
    cache_path=PATHWAY_ENRICHMENT_CACHE,
)
pathway_correlation = build_pathway_correlation(trait_gene_weights, gene_to_pathways)
print(f"Pathway correlation matrix shape: {pathway_correlation.shape}")
export_matrix_csv(pathway_correlation, GLOBAL_MATRIX_DIR / "pathway_correlation.csv")

# Gene-level correlation.
gene_level_correlations = build_gene_level_correlation(MAGMA_GENE_ZSTAT_FILE, TRAITS)
print(f"Gene-level correlation matrix shape: {gene_level_correlations.shape}")
export_matrix_csv(gene_level_correlations, GLOBAL_MATRIX_DIR / "gene_level_correlation.csv")


Pathway correlation matrix shape: (73, 73)


Gene-level correlation matrix shape: (73, 73)


{'csv': PosixPath('correlation_analysis_outputs/global_matrices/gene_level_correlation.csv'),
 'tsv': PosixPath('correlation_analysis_outputs/global_matrices/gene_level_correlation.tsv')}

## Run all dataset/folder analyses


In [18]:
dataset_results: Dict[str, Dict[str, object]] = {}
all_bootstrap_summaries = []
all_scatter_summaries = []
all_subset_correlation_statistics = []

for dataset_config in DATASET_CONFIGS:
    result = run_dataset_analysis(
        dataset_config=dataset_config,
        traits=TRAITS,
        genet_cor=genet_cor,
        genet_cor_pval=genet_cor_pval,
        gene_level_correlation=gene_level_correlations,
        pathway_correlation=pathway_correlation,
        subset_traits=SUBSET_TRAITS,
        output_root=OUTPUT_ROOT,
    )
    dataset_results[dataset_config.name] = result
    all_bootstrap_summaries.append(result["bootstrap_summary"])
    all_scatter_summaries.append(result["scatter_summary"])
    all_subset_correlation_statistics.append(result["subset_correlation_df"])

all_bootstrap_summaries = pd.concat(all_bootstrap_summaries, ignore_index=True)
all_scatter_summaries = pd.concat(all_scatter_summaries, ignore_index=True)
all_subset_correlation_statistics = pd.concat(all_subset_correlation_statistics, ignore_index=True)

export_dataframe_csv(
    all_bootstrap_summaries,
    OUTPUT_ROOT / "all_datasets.bootstrap_summary.csv",
    index=False,
)
export_dataframe_csv(
    all_scatter_summaries,
    OUTPUT_ROOT / "all_datasets.functional_vs_comparators_5panel.csv",
    index=False,
)
export_dataframe_csv(
    all_subset_correlation_statistics,
    OUTPUT_ROOT / "all_datasets.subset_functional_correlation_mc_pvals.csv",
    index=False,
)

all_bootstrap_summaries



=== tms_facs ===
Score folder: /mnt/shared-workspace/scdrsfm/results/real/tms_facs
Traits used for matrix analyses: 73



=== ts_facs ===
Score folder: /mnt/shared-workspace/scdrsfm/results/real/ts_facs
Traits used for matrix analyses: 73



=== tms_droplet ===
Score folder: /mnt/shared-workspace/scdrsfm/results/real/tms_droplet
Traits used for matrix analyses: 73


,dataset,target,metric,comparator,baseline,n_traits,n_sampled_traits,bootstrap_iterations_requested,mean,ci_lower,ci_upper,n
0,tms_facs,functional,pearson_r,genetic,0.215521,73,58,1000,0.215931,1.578471e-01,0.276130,1000
1,tms_facs,functional,pearson_r,abs_genetic,0.355825,73,58,1000,0.356590,3.099738e-01,0.406690,1000
2,tms_facs,functional,pearson_r,gene,0.423466,73,58,1000,0.424571,3.773212e-01,0.476795,1000
3,tms_facs,functional,pearson_r,pathway,0.589659,73,58,1000,0.589956,5.420841e-01,0.639798,1000
4,tms_facs,functional,pearson_r,marginal,0.800768,73,58,1000,0.800677,7.763230e-01,0.825344,1000
5,tms_facs,marginal,pearson_r,genetic,0.152440,73,58,1000,0.152912,1.063028e-01,0.208030,1000
6,tms_facs,marginal,pearson_r,abs_genetic,0.286903,73,58,1000,0.287670,2.477700e-01,0.325984,1000
7,tms_facs,marginal,pearson_r,gene,0.339412,73,58,1000,0.340846,3.006413e-01,0.378140,1000
8,tms_facs,marginal,pearson_r,pathway,0.632548,73,58,1000,0.633079,5.922195e-01,0.676280,1000
9,tms_facs,functional,partial_r2,genetic,0.001003,73,58,1000,0.001094,6.599640e-05,0.002721,1000


## Cross-dataset functional-correlation comparison


In [19]:
cross_dataset_summary = plot_cross_dataset_functional_correlations(
    dataset_results,
    reference_dataset="tms_facs",
    out_path=OUTPUT_ROOT / "cross_dataset_functional_correlation.png",
)
export_dataframe_csv(
    cross_dataset_summary,
    OUTPUT_ROOT / "cross_dataset_functional_correlation.csv",
    index=False,
)
cross_dataset_summary


,reference_dataset,comparison_dataset,n_traits,n_pairs,pearson_r,pearson_p
0,tms_facs,ts_facs,73,2628,0.570873,3.583191e-227
1,tms_facs,tms_droplet,73,2628,0.648456,1.881036e-313


## Final output index and CSV verification

In [20]:
# Verify that the expected CSV matrices and statistics were generated.
expected_csv_files = [
    GLOBAL_MATRIX_DIR / "genetic_correlation.csv",
    GLOBAL_MATRIX_DIR / "genetic_correlation_pvalues.csv",
    GLOBAL_MATRIX_DIR / "gene_level_correlation.csv",
    GLOBAL_MATRIX_DIR / "pathway_correlation.csv",
    GLOBAL_MATRIX_DIR / "traits.csv",
    OUTPUT_ROOT / "all_datasets.bootstrap_summary.csv",
    OUTPUT_ROOT / "all_datasets.functional_vs_comparators_5panel.csv",
    OUTPUT_ROOT / "all_datasets.subset_functional_correlation_mc_pvals.csv",
    OUTPUT_ROOT / "cross_dataset_functional_correlation.csv",
]

for dataset_config in DATASET_CONFIGS:
    dataset = dataset_config.name
    out_dir = OUTPUT_ROOT / dataset
    expected_csv_files.extend([
        out_dir / f"{dataset}.functional_correlation.csv",
        out_dir / f"{dataset}.marginal_correlation.csv",
        out_dir / f"{dataset}.genetic_correlation.csv",
        out_dir / f"{dataset}.genetic_correlation_pvalues.csv",
        out_dir / f"{dataset}.gene_level_correlation.csv",
        out_dir / f"{dataset}.pathway_correlation.csv",
        out_dir / f"{dataset}.bootstrap_summary.csv",
        out_dir / f"{dataset}.functional_vs_comparators_5panel.csv",
        out_dir / f"{dataset}.subset_functional_correlation_mc_pvals.csv",
        out_dir / f"{dataset}.subset_functional_correlation_matrix.csv",
        out_dir / f"{dataset}.subset_functional_mc_pvalue_matrix.csv",
    ])

missing_csv_files = [path for path in expected_csv_files if not path.is_file()]
if missing_csv_files:
    formatted = "\n".join(f"- {path}" for path in missing_csv_files)
    raise FileNotFoundError(f"Expected CSV outputs were not generated:\n{formatted}")

# Write a CSV manifest of every generated file, including the manifest itself.
manifest_path = OUTPUT_ROOT / "generated_files.csv"
generated_files = {str(path) for path in OUTPUT_ROOT.rglob("*") if path.is_file()}
generated_files.add(str(manifest_path))
generated_files_df = pd.DataFrame({"file": sorted(generated_files)})
generated_files_df.to_csv(manifest_path, index=False)

print(f"Verified {len(expected_csv_files)} expected CSV outputs.")
generated_files_df


Verified 42 expected CSV outputs.


,file
0,correlation_analysis_outputs/all_datasets.boot...
1,correlation_analysis_outputs/all_datasets.boot...
2,correlation_analysis_outputs/all_datasets.func...
3,correlation_analysis_outputs/all_datasets.func...
4,correlation_analysis_outputs/all_datasets.subs...
...,...
91,correlation_analysis_outputs/ts_facs/ts_facs.s...
92,correlation_analysis_outputs/ts_facs/ts_facs.s...
93,correlation_analysis_outputs/ts_facs/ts_facs.s...
94,correlation_analysis_outputs/ts_facs/ts_facs.s...
